In [1]:
#SETUP DOCKER+LANGFUSE

#!cd ~/Desktop/Mini-Project-LangFuse/langfuse
#!docker compose up -d
#http://localhost:3000/

In [3]:
#SETUP LLM+LANGFUSE

#Import keys
import os
import json
from dotenv import load_dotenv
load_dotenv()

#Set up the LLM client
from openai import OpenAI
client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY_1"),
    base_url="https://api.groq.com/openai/v1"
)

#Connect LangFuse
from langfuse import observe, Langfuse
langfuse = Langfuse()

#Models
BASELINE_MODEL = "llama-3.1-8b-instant"
NEW_MODEL = "llama-3.3-70b-versatile"

Baseline model: llama-3.1-8b-instant
New model: llama-3.3-70b-versatile
LangFuse: Connected


/Users/pedroalexleite/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [4]:
#SIMPLE CHATBOT

#Prompt
SYSTEM_PROMPT = """You are a helpful healthcare assistant. You provide accurate, 
general health information to help users understand medical topics.

Rules:
- Always recommend consulting a doctor for specific medical advice.
- Be clear and concise.
- If you don't know something, say so honestly.
- Never diagnose conditions or prescribe medication.
"""

#ChatBot
@observe()
def healthcare_chatbot(user_question, model=BASELINE_MODEL):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_question}
        ]
    )
    return response.choices[0].message.content

In [5]:
#LLM AS A JUDGE

#Prompt
JUDGE_PROMPT = """You are an expert evaluator for a healthcare chatbot. 
Compare the actual answer to the expected answer and score the actual answer.

Question: {question}

Expected Answer: {expected_answer}

Actual Answer: {actual_answer}

Score the actual answer from 1 to 5:
1 = Completely wrong or harmful
2 = Mostly incorrect or missing key information
3 = Partially correct but incomplete
4 = Mostly correct with minor omissions
5 = Fully correct and complete

Respond with ONLY a JSON object in this exact format, nothing else:
{{"score": <number>, "reasoning": "<brief explanation>"}}
"""

#Judge
@observe()
def judge_answer(question, expected_answer, actual_answer, model=BASELINE_MODEL):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": JUDGE_PROMPT.format(
                question=question,
                expected_answer=expected_answer,
                actual_answer=actual_answer
            )}
        ]
    )
    return json.loads(response.choices[0].message.content)

In [6]:
#GOLDEN DATASET (20 Healthcare question/answer pairs across 7 categories)

with open("../Data/golden_dataset.json", "r") as f:
    golden_dataset = json.load(f)

print(f"Loaded {len(golden_dataset)} questions")

Loaded 20 questions


In [9]:
#EVALUATE CHATBOT ANSWERS USING LLM AS A JUDGE

def run_evaluation(model_name):
    print(f"\n{'='*50}")
    print(f"Evaluating model: {model_name}")
    print(f"{'='*50}\n")
    
    results = []
    
    for item in golden_dataset:
        print(f"[{item['id']}/20] {item['question'][:55]}...")
        
        #Chatbot answer
        answer = healthcare_chatbot(item["question"], model=model_name)
        
        #Judge the answer
        eval_result = judge_answer(
            item["question"], 
            item["expected_answer"], 
            answer, 
            model=BASELINE_MODEL  
        )

        #Append results
        results.append({
            "id": item["id"],
            "category": item["category"],
            "question": item["question"],
            "expected_answer": item["expected_answer"],
            "actual_answer": answer,
            "score": eval_result["score"],
            "reasoning": eval_result["reasoning"]
        })
        
        print(f"  Score: {eval_result['score']} — {eval_result['reasoning'][:70]}")
    
    avg_score = sum(r["score"] for r in results) / len(results)
    print(f"\n{'='*50}")
    print(f"Model: {model_name}")
    print(f"Average Score: {avg_score:.2f} / 5.00")
    print(f"{'='*50}")
    
    return results, avg_score

baseline_results, baseline_score = run_evaluation(BASELINE_MODEL)
new_results, new_score = run_evaluation(NEW_MODEL)


Evaluating model: llama-3.1-8b-instant

[1/20] What are the common side effects of ibuprofen?...
  Score: 4 — The actual answer is correct about ibuprofen side effects, including c
[2/20] What are the side effects of amoxicillin?...
  Score: 4 — The actual answer provides a comprehensive list of potential side effe
[3/20] Can I take ibuprofen while pregnant?...
  Score: 4 — The actual answer is mostly correct with minor omissions about the spe
[4/20] Is it safe to mix paracetamol and ibuprofen?...
  Score: 5 — The actual answer accurately conveys the guidelines for safely taking 
[5/20] Can I drink alcohol while taking antibiotics?...
  Score: 4 — The actual answer provides accurate information on the general recomme
[6/20] What's the difference between a cold and the flu?...
  Score: 5 — The actual answer provides a detailed and accurate comparison between 
[7/20] What are the symptoms of diabetes?...
  Score: 4 — The actual answer covers the common symptoms of diabetes correctly and

In [10]:
#COMPARE MODELS

def compare_results(baseline_results, baseline_score, new_results, new_score):
    baseline_model = BASELINE_MODEL
    new_model = NEW_MODEL
    
    print(f"\n{'='*50}")
    print(f"COMPARISON")
    print(f"{'='*50}")
    print(f"Baseline ({baseline_model}): {baseline_score:.2f}")
    print(f"New model ({new_model}): {new_score:.2f}")
    print(f"Difference: {new_score - baseline_score:+.2f}")
    
    #Per-question breakdown
    print(f"\nPer-question comparison:")
    improved, same, worse = 0, 0, 0
    for b, n in zip(baseline_results, new_results):
        diff = n["score"] - b["score"]
        if diff > 0:
            indicator = "✅"
            improved += 1
        elif diff < 0:
            indicator = "❌"
            worse += 1
        else:
            indicator = "➡️"
            same += 1
        print(f"  {indicator} Q{b['id']}: {b['score']}→{n['score']} ({diff:+d}) {b['question'][:50]}")
    
    #Per-category breakdown
    from collections import defaultdict
    baseline_by_cat = defaultdict(list)
    new_by_cat = defaultdict(list)
    for b, n in zip(baseline_results, new_results):
        baseline_by_cat[b["category"]].append(b["score"])
        new_by_cat[n["category"]].append(n["score"])
    
    print(f"\nPer-category comparison:")
    for cat in baseline_by_cat:
        b_avg = sum(baseline_by_cat[cat]) / len(baseline_by_cat[cat])
        n_avg = sum(new_by_cat[cat]) / len(new_by_cat[cat])
        diff = n_avg - b_avg
        indicator = "✅" if diff > 0 else "❌" if diff < 0 else "➡️"
        print(f"  {indicator} {cat}: {b_avg:.2f}→{n_avg:.2f} ({diff:+.2f})")
    
    #Summary
    print(f"\nSummary: {improved} improved, {same} same, {worse} worse")xw

compare_results(baseline_results, baseline_score, new_results, new_score)


COMPARISON
Baseline (llama-3.1-8b-instant): 4.25
New model (llama-3.3-70b-versatile): 4.35
Difference: +0.10

Per-question comparison:
  ➡️ Q1: 4→4 (+0) What are the common side effects of ibuprofen?
  ➡️ Q2: 4→4 (+0) What are the side effects of amoxicillin?
  ✅ Q3: 4→5 (+1) Can I take ibuprofen while pregnant?
  ➡️ Q4: 5→5 (+0) Is it safe to mix paracetamol and ibuprofen?
  ➡️ Q5: 4→4 (+0) Can I drink alcohol while taking antibiotics?
  ❌ Q6: 5→4 (-1) What's the difference between a cold and the flu?
  ✅ Q7: 4→5 (+1) What are the symptoms of diabetes?
  ➡️ Q8: 4→4 (+0) What are the warning signs of a heart attack?
  ➡️ Q9: 4→4 (+0) How much water should I drink per day?
  ✅ Q10: 4→5 (+1) How many hours of sleep do adults need?
  ➡️ Q11: 4→4 (+0) What is a normal blood pressure reading?
  ➡️ Q12: 4→4 (+0) What foods should I eat to lower cholesterol?
  ➡️ Q13: 4→4 (+0) How much exercise should I get per week?
  ✅ Q14: 4→5 (+1) What are the signs of depression?
  ➡️ Q15: 4→4 (+0) How 